# Day 11 — Pandas II: Cleaning, GroupBy & Merge
Cleaning and reshaping messy data (raw → clean).

## 1. Create a Messy Dataset

In [1]:
import pandas as pd
import numpy as np

raw_data = {
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Ali Khan", "sara ahmed", "Bilal  Raza", None, "Ayesha", "Zain Malik", "Fatima", "Hassan"],
    "signup_date": ["2023-01-15", "2023/02/20", "15-03-2023", "2023-04-10",
                     None, "2023-06-05", "2023-07-01", "not available"],
    "purchase_amount": ["250", "300.5", None, "150", "400", "abc", "220", "310"],
    "city": ["Karachi", "Lahore", "karachi", "Islamabad", "Lahore", None, "Karachi", "Lahore"],
}

df_raw = pd.DataFrame(raw_data)
df_raw

,customer_id,name,signup_date,purchase_amount,city
0,1,Ali Khan,2023-01-15,250,Karachi
1,2,sara ahmed,2023/02/20,300.5,Lahore
2,3,Bilal Raza,15-03-2023,None,karachi
3,4,None,2023-04-10,150,Islamabad
4,5,Ayesha,None,400,Lahore
5,6,Zain Malik,2023-06-05,abc,None
6,7,Fatima,2023-07-01,220,Karachi
7,8,Hassan,not available,310,Lahore


## 2. Inspect Missing & Problematic Data

In [2]:
print("Missing values per column:")
print(df_raw.isna().sum())

print("\nData types:")
print(df_raw.dtypes)

Missing values per column:
customer_id        0
name               1
signup_date        1
purchase_amount    1
city               1
dtype: int64

Data types:
customer_id         int64
name               object
signup_date        object
purchase_amount    object
city               object
dtype: object


## 3. Clean the Name Column

In [3]:
df = df_raw.copy()

df["name"] = df["name"].str.strip().str.title()
df["name"] = df["name"].fillna("Unknown")

df[["customer_id", "name"]]

,customer_id,name
0,1,Ali Khan
1,2,Sara Ahmed
2,3,Bilal Raza
3,4,Unknown
4,5,Ayesha
5,6,Zain Malik
6,7,Fatima
7,8,Hassan


## 4. Clean the Date Column
Convert mixed date formats into proper datetime objects.

In [4]:
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

df[["customer_id", "signup_date"]]

,customer_id,signup_date
0,1,2023-01-15
1,2,NaT
2,3,NaT
3,4,2023-04-10
4,5,NaT
5,6,2023-06-05
6,7,2023-07-01
7,8,NaT


## 5. Clean the Purchase Amount Column

In [5]:
df["purchase_amount"] = pd.to_numeric(df["purchase_amount"], errors="coerce")

mean_amount = df["purchase_amount"].mean()
df["purchase_amount"] = df["purchase_amount"].fillna(round(mean_amount, 2))

df[["customer_id", "purchase_amount"]]

,customer_id,purchase_amount
0,1,250.00
1,2,300.50
2,3,271.75
3,4,150.00
4,5,400.00
5,6,271.75
6,7,220.00
7,8,310.00


## 6. Clean the City Column

In [6]:
df["city"] = df["city"].str.strip().str.title()
df["city"] = df["city"].fillna("Unknown")

df[["customer_id", "city"]]

,customer_id,city
0,1,Karachi
1,2,Lahore
2,3,Karachi
3,4,Islamabad
4,5,Lahore
5,6,Unknown
6,7,Karachi
7,8,Lahore


## 7. Drop Rows That Are Still Unusable
Any row missing a signup date after cleaning is dropped, since that field can't be reasonably guessed.

In [7]:
before_count = len(df)
df = df.dropna(subset=["signup_date"])
after_count = len(df)

print(f"Dropped {before_count - after_count} row(s) with unusable signup dates.")
df

Dropped 4 row(s) with unusable signup dates.


,customer_id,name,signup_date,purchase_amount,city
0,1,Ali Khan,2023-01-15,250.00,Karachi
3,4,Unknown,2023-04-10,150.00,Islamabad
5,6,Zain Malik,2023-06-05,271.75,Unknown
6,7,Fatima,2023-07-01,220.00,Karachi


## 8. GroupBy & Aggregation

In [8]:
city_summary = df.groupby("city")["purchase_amount"].agg(["mean", "sum", "count"])
city_summary

,mean,sum,count
city,,,
Islamabad,150.00,150.00,1
Karachi,235.00,470.00,2
Unknown,271.75,271.75,1


## 9. Pivot Table

In [9]:
df["signup_month"] = df["signup_date"].dt.month_name()

pivot = df.pivot_table(
    values="purchase_amount",
    index="city",
    columns="signup_month",
    aggfunc="sum",
    fill_value=0
)
pivot

signup_month,April,January,July,June
city,,,,
Islamabad,150.0,0.0,0.0,0.00
Karachi,0.0,250.0,220.0,0.00
Unknown,0.0,0.0,0.0,271.75


## 10. Merge / Join with a Second Table

In [10]:
loyalty_data = pd.DataFrame({
    "customer_id": [1, 2, 3, 5, 6, 7],
    "loyalty_tier": ["Gold", "Silver", "Bronze", "Gold", "Silver", "Bronze"],
})

merged = df.merge(loyalty_data, on="customer_id", how="left")
merged["loyalty_tier"] = merged["loyalty_tier"].fillna("Not Enrolled")

merged[["customer_id", "name", "city", "loyalty_tier"]]

,customer_id,name,city,loyalty_tier
0,1,Ali Khan,Karachi,Gold
1,4,Unknown,Islamabad,Not Enrolled
2,6,Zain Malik,Unknown,Silver
3,7,Fatima,Karachi,Bronze


## 11. Concat Example

In [11]:
new_customers = pd.DataFrame({
    "customer_id": [9, 10],
    "name": ["Omar Sheikh", "Nida Farooq"],
    "signup_date": pd.to_datetime(["2023-08-01", "2023-08-15"]),
    "purchase_amount": [275.0, 190.0],
    "city": ["Karachi", "Lahore"],
    "signup_month": ["August", "August"],
})

combined = pd.concat([df, new_customers], ignore_index=True)
combined.tail()

,customer_id,name,signup_date,purchase_amount,city,signup_month
1,4,Unknown,2023-04-10,150.00,Islamabad,April
2,6,Zain Malik,2023-06-05,271.75,Unknown,June
3,7,Fatima,2023-07-01,220.00,Karachi,July
4,9,Omar Sheikh,2023-08-01,275.00,Karachi,August
5,10,Nida Farooq,2023-08-15,190.00,Lahore,August


## 12. apply / map

In [12]:
# apply: row-wise custom logic
def classify_spender(amount):
    if amount >= 300:
        return "High"
    elif amount >= 200:
        return "Medium"
    else:
        return "Low"

df["spender_tier"] = df["purchase_amount"].apply(classify_spender)

# map: simple value substitution
city_codes = {"Karachi": "KHI", "Lahore": "LHE", "Islamabad": "ISB", "Unknown": "UNK"}
df["city_code"] = df["city"].map(city_codes)

df[["customer_id", "purchase_amount", "spender_tier", "city", "city_code"]]

,customer_id,purchase_amount,spender_tier,city,city_code
0,1,250.00,Medium,Karachi,KHI
3,4,150.00,Low,Islamabad,ISB
5,6,271.75,Medium,Unknown,UNK
6,7,220.00,Medium,Karachi,KHI


## 13. Final Clean Dataset

In [13]:
df

,customer_id,name,signup_date,purchase_amount,city,signup_month,spender_tier,city_code
0,1,Ali Khan,2023-01-15,250.00,Karachi,January,Medium,KHI
3,4,Unknown,2023-04-10,150.00,Islamabad,April,Low,ISB
5,6,Zain Malik,2023-06-05,271.75,Unknown,June,Medium,UNK
6,7,Fatima,2023-07-01,220.00,Karachi,July,Medium,KHI
